In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

# Standardized Paths
INPUT_FILE = '../resultados/comparativo_perfil_cenarios_qgis.csv'
OUTPUT_PDF = '../resultados/figure-net-behavioral-shift-heatmap-fig13.pdf'
OUTPUT_PNG = '../resultados/figure-net-behavioral-shift-heatmap-fig13.png'  # For display purposes

def natural_sort_key(s):
    """Sort scenarios (CI, CII, CIII... CXVIII) in natural order."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

def generate_behavioral_impact_map():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found.")
        return

    # 1. Load data (Using ';' separator as defined in GAMA)
    df = pd.read_csv(INPUT_FILE, sep=',')
    
    # 2. Extract Baseline and Totals
    unique_scenarios = sorted(df['NM_CENARIO'].unique(), key=natural_sort_key)
    total_agents = len(df[df['NM_CENARIO'] == unique_scenarios[0]])
    
    # Baseline distribution (Initial profile count)
    initial_dist = df[df['NM_CENARIO'] == unique_scenarios[0]]['TP_COMPORTAMENTO'].value_counts()
    
    # 3. Process each scenario to calculate Net Change
    profiles = ['AMBIENTALISTA', 'MODERADO', 'PERDULARIO']
    impact_data = []

    for scenario in unique_scenarios:
        df_scen = df[df['NM_CENARIO'] == scenario]
        final_dist = df_scen['TP_NOVO_COMPORTAMENTO'].value_counts()
        
        row = {'Scenario': scenario}
        for p in profiles:
            # Net change as percentage of the total household population
            net_change = ((final_dist.get(p, 0) - initial_dist.get(p, 0)) / total_agents) * 100
            row[p] = net_change
        impact_data.append(row)

    # Convert to DataFrame for heatmap
    impact_df = pd.DataFrame(impact_data).set_index('Scenario')

    # 4. Plotting (Elsevier-ready styles)
    plt.figure(figsize=(10, 12))
    sns.set_theme(style="white")
    
    # RdBu_r: Red for increase (positive), Blue for decrease (negative)
    # Center=0 ensures white is neutral (no change)
    ax = sns.heatmap(
        impact_df, 
        annot=True, 
        fmt=".2f", 
        cmap="RdBu", 
        center=0,
        linewidths=.5,
        cbar_kws={'label': 'Net Change (% of Households)'}
    )
    
    plt.title("Change in consumption profile based on a simulation scenario.\n(Comparison: Simulated Final State vs. Baseline)", 
              fontsize=14, fontweight='bold', pad=20)
    plt.xlabel("Consumer Behavioral Profile", fontsize=12)
    plt.ylabel("Scenario ID", fontsize=12)
    
    # Standardizing profile labels for the article
    ax.set_xticklabels(['Environmentalist', 'Moderate', 'Wasteful'], fontsize=11)
    
    plt.tight_layout()
    
    # 5. Saving (PDF as primary, PNG for preview)
    plt.savefig(OUTPUT_PDF, format='pdf', bbox_inches='tight')
    plt.savefig(OUTPUT_PNG, format='png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Success! Map saved as PDF: {OUTPUT_PDF}")

if __name__ == "__main__":
    generate_behavioral_impact_map()

Success! Map saved as PDF: ../resultados/figure-net-behavioral-shift-heatmap-fig13.pdf
